# Run histology pipeline on Google Drive dataset (no local storage)

This notebook runs the full 6-step pipeline in **Google Colab** so you don't need disk space on your PC.

**Your setup:** Dataset = **Meta_MasterThesis.tar** in My Drive. Repo = **amernajdawi/histology**. Defaults are set; just run the cells in order.

**Before you start:**
1. (Optional) **FastGlioma checkpoint** for Step 1: upload to Drive (e.g. `My Drive/colab_ckpts/`) if you have it.
2. **Dataset in Drive:** Run **Section 2** to mount Drive and use your .tar from My Drive (path already set to `/content/drive/MyDrive`).

## 1. Download dataset with gdown (recommended – faster)

In [ ]:
# Dataset file ID from your share link
GDOWN_FILE_ID = "1G1SkbhvfJVKIbTDYqUjJ9cdkwtHTNLnU"
WORK_DIR = "/content/work"

!pip install gdown -q
from pathlib import Path
import zipfile
import os

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
zip_path = WORK_DIR + "/dataset.zip"

print("Downloading dataset...")
# Use ! so you see gdown output; for large files it may print a link - open it, confirm, then re-run this cell
!gdown {GDOWN_FILE_ID} -O "{zip_path}"

if not Path(zip_path).exists():
    print("\n*** Download failed. Do ONE of the following:")
    print("1. In Google Drive: right-click the file -> Share -> change to 'Anyone with the link' (Viewer), then re-run this cell.")
    print("2. Or add the file to My Drive, then SKIP this section and use Section 2 (Optional Use Google Drive) and set DRIVE_DATASET_PATH to your folder.")
    raise SystemExit(1)

print("Extracting...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(WORK_DIR)
BASE_DIR = WORK_DIR
print("Done. Dataset at", BASE_DIR)

In [ ]:
# List extracted contents (dataset folders should appear here)
print("Contents of", WORK_DIR, ":")
for p in sorted(Path(WORK_DIR).iterdir()):
    print(" ", p.name)

## 2. (Optional) Use Google Drive instead of gdown

**If you previously got "No space left on device":** Run the next cell once to free Colab disk, then run the Drive cell below. The notebook now extracts the .tar **to Google Drive** (not Colab), so you won't run out of space.

In [ ]:
# Optional: run this ONCE if you got 'No space left on device' — frees Colab disk for a fresh run
!rm -rf /content/work/* 2>/dev/null; echo 'Freed /content/work. Now run the cell below.'


In [ ]:
# Run this cell to use your dataset FROM DRIVE (Meta_MasterThesis.tar in My Drive).
from pathlib import Path
import zipfile
import tarfile
WORK_DIR = "/content/work"
from google.colab import drive
drive.mount('/content/drive')

# Path to folder that contains your data: My Drive root (Meta_MasterThesis.tar is here)
DRIVE_DATASET_PATH = "/content/drive/MyDrive"
DRIVE_DATASET_ZIP = None  # or a .zip path to extract

# When .tar is too big for Colab disk (~45 GB), we extract TO DRIVE (uses your Drive space, not Colab).
EXTRACT_TO_DRIVE_PATH = "/content/drive/MyDrive/extracted_meta"  # folder on Drive for extracted .tar

drive_path = Path(DRIVE_DATASET_PATH)
if not drive_path.exists():
    print("Path not found. Listing contents of My Drive:")
    my_drive = Path("/content/drive/MyDrive")
    if my_drive.exists():
        for p in sorted(my_drive.iterdir()):
            print(" ", p.name)
        print('\nSet DRIVE_DATASET_PATH to the correct path and re-run.')
    BASE_DIR = DRIVE_DATASET_PATH
else:
    tar_files = list(drive_path.glob("*.tar"))
    import shutil
    free_gb = shutil.disk_usage('/content').free / (1024**3)
    already_extracted = Path(EXTRACT_TO_DRIVE_PATH)
    if already_extracted.exists() and any(already_extracted.iterdir()):
        print("Using already-extracted folder on Drive:", EXTRACT_TO_DRIVE_PATH)
        BASE_DIR = EXTRACT_TO_DRIVE_PATH
    elif tar_files and len(tar_files) == 1:
        tar_path = tar_files[0]
        tar_size_gb = tar_path.stat().st_size / (1024**3)
        # Extract to DRIVE so we don't fill Colab disk (81 GB won't fit in 45 GB)
        print("Found .tar:", tar_path.name, f"({tar_size_gb:.1f} GB). Colab free: {free_gb:.1f} GB.")
        print("Extracting to Google Drive (this uses your Drive space, not Colab). May take 30–90 min...")
        already_extracted.mkdir(parents=True, exist_ok=True)
        with tarfile.open(tar_path, 'r:*') as tf:
            tf.extractall(EXTRACT_TO_DRIVE_PATH, filter='data')
        BASE_DIR = EXTRACT_TO_DRIVE_PATH
        print("Done. Dataset at", BASE_DIR)
    elif DRIVE_DATASET_ZIP:
        with zipfile.ZipFile(DRIVE_DATASET_ZIP, 'r') as z:
            z.extractall(WORK_DIR)
        BASE_DIR = WORK_DIR
        print("Extracted to", BASE_DIR)
    else:
        BASE_DIR = DRIVE_DATASET_PATH
        print("Using Drive folder:", BASE_DIR)

In [ ]:
# Skip the cell below if you used gdown (Section 1). Run it only if you extracted a ZIP from Drive.

In [ ]:
# (Only if you used Drive ZIP above) List extracted dataset folders:
for p in sorted(Path(WORK_DIR).iterdir()):
    print(p.name)

## 3. Clone the pipeline repo and install FastGlioma

In [ ]:
# Clone the FULL FastGlioma repo (needed for Step 1 imports)
# Reset to /content so shell cwd is valid (avoids 'getcwd: cannot access parent directories')
%cd /content
!rm -rf /content/fastglioma_repo
!git clone https://github.com/MLNeurosurg/fastglioma.git /content/fastglioma_repo
# If this fails with "Getting requirements to build wheel", run the NEXT cell (fallback) instead.
!pip install -e /content/fastglioma_repo

In [ ]:
# Clone YOUR pipeline repo (replace with your GitHub URL if you pushed it)
# If you don't have it on GitHub: upload the repo ZIP to Drive and use the next cell instead.
REPO_URL = "https://github.com/amernajdawi/histology.git"  # this repo

import subprocess
try:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/new_his"], capture_output=True, text=True)
    if r.returncode != 0:
        print("Clone failed:", r.stderr or r.stdout)
        PIPELINE_DIR = None
    else:
        PIPELINE_DIR = "/content/new_his"
        print("Cloned to", PIPELINE_DIR)
except Exception as e:
    print("Clone error:", e)
    PIPELINE_DIR = None  # Will use ZIP from Drive

**If the cell above failed** with "Getting requirements to build wheel" or "subprocess-exited-with-error": FastGlioma's `setup.py` pins old package versions that often don't work on Colab's Python. Run the cell below to install only the dependencies (no version pins). We'll use the FastGlioma **source code** from the clone (copied into the pipeline in the next section); the pipeline does not require the package to be installed.

In [ ]:
# Fallback: install FastGlioma dependencies only (Colab-compatible, no strict version pins)
!pip install torch torchvision timm pytorch-lightning pandas pyyaml tqdm scikit-image opencv-python tifffile scikit-learn matplotlib huggingface-hub

In [ ]:
# If you uploaded the pipeline as ZIP to Drive, set this and run:
from pathlib import Path
PIPELINE_ZIP_ON_DRIVE = None  # e.g. "/content/drive/MyDrive/new_his.zip"

if PIPELINE_ZIP_ON_DRIVE:
    import zipfile
    with zipfile.ZipFile(PIPELINE_ZIP_ON_DRIVE, 'r') as z:
        z.extractall("/content")
    # Often the zip has a top-level folder 'new_his'
    if (Path("/content/new_his") / "flexible_pipeline").exists():
        PIPELINE_DIR = "/content/new_his"
    else:
        PIPELINE_DIR = "/content"  # adjust if your zip structure differs
    print("Pipeline extracted to", PIPELINE_DIR)

if not PIPELINE_DIR or not Path(PIPELINE_DIR).exists():
    print("PIPELINE_DIR not set. Either set REPO_URL or PIPELINE_ZIP_ON_DRIVE.")
else:
    import sys
    sys.path.insert(0, PIPELINE_DIR)
    %cd $PIPELINE_DIR
    print("Working in", PIPELINE_DIR)

In [ ]:
# Point pipeline's fastglioma to the full repo we installed
import shutil
from pathlib import Path
if PIPELINE_DIR is None:
    if Path("/content/new_his").exists():
        PIPELINE_DIR = "/content/new_his"
        print("Using PIPELINE_DIR = /content/new_his (from previous run).")
    else:
        raise RuntimeError(
            "PIPELINE_DIR is not set. Run the cells above: first clone the pipeline repo (or set PIPELINE_ZIP_ON_DRIVE), then the cell that sets 'Working in'."
        )
pd = Path(PIPELINE_DIR)
if (pd / "fastglioma").exists():
    # Replace stub with full package: copy from cloned repo
    shutil.rmtree(pd / "fastglioma", ignore_errors=True)
    shutil.copytree("/content/fastglioma_repo/fastglioma", pd / "fastglioma")
    print("Replaced fastglioma stub with full package.")
else:
    import sys
    sys.path.insert(0, "/content/fastglioma_repo")
    print("Using FastGlioma from /content/fastglioma_repo")

In [ ]:
!pip install pydicom scikit-image -q

## 4. Checkpoint (for Step 1)
Mount Drive if your checkpoint (or pipeline ZIP) is there. Then copy the checkpoint into the project.

In [ ]:
# Mount Drive first if you use checkpoint or pipeline ZIP from Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ckpt_drive = "/content/drive/MyDrive/colab_ckpts/fastglioma_highres_model.ckpt"  # <-- Put your ckpt here in Drive
ckpt_dest = Path(PIPELINE_DIR) / "fastglioma_ckpts" / "fastglioma_highres_model.ckpt"
Path(ckpt_dest).parent.mkdir(parents=True, exist_ok=True)
if Path(ckpt_drive).exists():
    shutil.copy(ckpt_drive, ckpt_dest)
    print("Checkpoint copied.")
else:
    print("Checkpoint not found at", ckpt_drive, "- Step 1 may fail unless you have it elsewhere.")

## 5. Run the pipeline (Steps 1–6)

**First run (10 samples for supervisor):** In the setup cell above, keep `MAX_SAMPLES = 10`. Run the setup cell, then Steps 1–6. Run Section 7 to view and download 10 sample images. Copy to Drive (Section 6) and share with your supervisor.

**Full dataset:** Set `MAX_SAMPLES = None` in the setup cell, re-run the setup cell, then re-run Steps 1–6. Run Section 6 to save all results to Drive.

In [ ]:
import os
%cd $PIPELINE_DIR

# Output dirs (under work dir so we don't fill Drive unless you copy them later)
os.makedirs(WORK_DIR + "/strips_heatmaps", exist_ok=True)
os.makedirs(WORK_DIR + "/overlaid_strips", exist_ok=True)
os.makedirs(WORK_DIR + "/combined_strips", exist_ok=True)
os.makedirs(WORK_DIR + "/gradcam_heatmaps", exist_ok=True)
os.makedirs(WORK_DIR + "/stitched_images", exist_ok=True)

fp = "flexible_pipeline"
# BASE_DIR may contain a single top-level folder (e.g. Meta_MasterThesis) that actually holds the datasets.
# If so, automatically step into that folder so step1 can find <dataset>/<series>/strips/*.dcm.
from pathlib import Path as _Path
_root = _Path(BASE_DIR)
_subdirs = [p for p in _root.iterdir() if p.is_dir() and not p.name.startswith('.')]
if len(_subdirs) == 1:
    base = str(_subdirs[0])
    print("Using inner dataset root:", base)
else:
    base = str(_root)
out1 = WORK_DIR + "/strips_heatmaps"
out2 = WORK_DIR + "/overlaid_strips"
out3 = WORK_DIR + "/combined_strips"
out4 = WORK_DIR + "/gradcam_heatmaps"
out5 = WORK_DIR + "/stitched_images"
ckpt = str(Path(PIPELINE_DIR) / "fastglioma_ckpts" / "fastglioma_highres_model.ckpt")

# Run on 10 samples first (for supervisor), then set to None and re-run Steps 1-6 for full dataset
MAX_SAMPLES = 10  # Set to None to process entire dataset
max_strips_arg = f"--max-strips {MAX_SAMPLES}" if MAX_SAMPLES else ""

In [ ]:
# Step 1: DICOM -> PNG (use MAX_SAMPLES=10 in cell above for quick run, then None for full)
!python {fp}/step1_convert_dicom_to_png.py --base-dir "{base}" --checkpoint "{ckpt}" --output-dir "{out1}" {max_strips_arg}

In [ ]:
# Step 2: Overlay duplicates (optional: add --manual-pairs path/to/config.json)
config_path = Path(PIPELINE_DIR) / "flexible_pipeline" / "example_config.json"
if config_path.exists():
    !python {fp}/step2_overlay_duplicates.py --input-dir "{out1}" --output-dir "{out2}" --manual-pairs "{config_path}"
else:
    !python {fp}/step2_overlay_duplicates.py --input-dir "{out1}" --output-dir "{out2}"

In [ ]:
# Step 3: Stitch groups
if config_path.exists():
    !python {fp}/step3_stitch_groups.py --input-dir "{out2}" --output-dir "{out3}" --config "{config_path}"
else:
    !python {fp}/step3_stitch_groups.py --input-dir "{out2}" --output-dir "{out3}"

In [ ]:
# Step 4: Reassemble Grad-CAM (script looks for dataset/series/patches under current dir)
import os
_orig = os.getcwd()
# Use the same dataset root as Step 1 (base), which points at the folder containing MUV_* subfolders.
os.chdir(base)
!python "{PIPELINE_DIR}/flexible_pipeline/step4_reassemble_gradcam.py" --output-dir "{out4}" --target-width 3600 --target-height 3900
os.chdir(_orig)

In [ ]:
# Step 5: Overlay Grad-CAM on stitched image (outputs are in combined_strips + gradcam_heatmaps)
import glob, subprocess
stitched_list = sorted(glob.glob(out3 + "/*_overlaid_strips_stitched.png"))
gradcam_list = sorted(glob.glob(out4 + "/*_gradcam_heatmap.png"))
if stitched_list and gradcam_list:
    for st, gc in zip(stitched_list, gradcam_list):
        subprocess.run(["python", f"{PIPELINE_DIR}/flexible_pipeline/step5_overlay_gradcam.py", "--stitched-image", st, "--gradcam-heatmap", gc, "--output-dir", out5], check=True)
else:
    !python "{PIPELINE_DIR}/flexible_pipeline/step5_overlay_gradcam.py" --output-dir "{out5}"

In [ ]:
# Step 6: Combine with original image. Patches are under BASE_DIR; pass --patches-dir so step6 finds them.
from pathlib import Path
patches_dir_step6 = None
for d in Path(BASE_DIR).iterdir():
    if d.is_dir() and not d.name.startswith('.'):
        for s in d.iterdir():
            if s.is_dir() and s.name.isdigit():
                pd = s / "patches"
                if pd.exists() and list(pd.glob("*ALA*.tif")):
                    patches_dir_step6 = str(pd)
                    break
        if patches_dir_step6:
            break
orig_img = None
for p in [Path(WORK_DIR) / "unnamed.png", Path(BASE_DIR) / "unnamed.png", Path(PIPELINE_DIR) / "unnamed.png"]:
    if p.exists():
        orig_img = str(p)
        break
cmd = ["python", f"{PIPELINE_DIR}/flexible_pipeline/step6_combine_with_original.py", "--output-path", f"{out5}/unnamed_with_gradcam.png"]
if patches_dir_step6:
    cmd += ["--patches-dir", patches_dir_step6]
if orig_img:
    cmd += ["--original-image", orig_img]
import subprocess
subprocess.run(cmd, check=True)

## 6. Save results back to Drive (optional)

In [ ]:
save_to_drive = True  # Set to True to copy outputs to Drive
drive_output = "/content/drive/MyDrive/pipeline_outputs"

if save_to_drive:
    from pathlib import Path
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    Path(drive_output).mkdir(parents=True, exist_ok=True)
    for name in ["strips_heatmaps", "overlaid_strips", "combined_strips", "gradcam_heatmaps", "stitched_images"]:
        src = Path(WORK_DIR) / name
        if src.exists():
            dest = Path(drive_output) / name
            if dest.exists():
                shutil.rmtree(dest)
            shutil.copytree(src, dest)
            print("Copied", name, "to Drive")
    print("Done. Outputs in", drive_output)

## 7. Where results are saved & how to view them (including 10 samples for supervisor)

In [ ]:
# WHERE RESULTS ARE SAVED:
# 1) In Colab (temporary): /content/work/strips_heatmaps, overlaid_strips, combined_strips, gradcam_heatmaps, stitched_images
# 2) In Google Drive (if you ran Section 6): My Drive / pipeline_outputs / (same folders)
# HOW TO SEE THEM: Run this cell; it lists paths and shows up to 10 sample images you can send to your supervisor.
from pathlib import Path
from IPython.display import Image, display
WORK_DIR = "/content/work"
print("=== Results in Colab (paths) ===")
for name in ["strips_heatmaps", "overlaid_strips", "combined_strips", "gradcam_heatmaps", "stitched_images"]:
    d = Path(WORK_DIR) / name
    if d.exists():
        files = list(d.glob("*.png"))
        print(f"  {d}: {len(files)} file(s)")
stitched_dir = Path(WORK_DIR) / "stitched_images"
with_gradcam = sorted(stitched_dir.glob("*_with_gradcam.png")) if stitched_dir.exists() else []
stitched_only = sorted(stitched_dir.glob("*_overlaid_strips_stitched.png")) if stitched_dir.exists() else []
print("\n=== Main outputs (best to share with supervisor) ===")
for f in (with_gradcam or stitched_only)[:20]:
    print(f"  {f.name}")
print("\n=== 10 sample images (for supervisor) ===")
samples = (with_gradcam or stitched_only)[:10]
if samples:
    for i, f in enumerate(samples, 1):
        print(f"  {i}. {f.name}")
        display(Image(str(f), width=500))
else:
    print("  No images yet. Run Steps 1-6 first.")
print("\n=== In Google Drive (after Section 6) ===")
print("  My Drive / pipeline_outputs / stitched_images  (download or share folder with supervisor)")